# 1 — Banded Attention NLP Ensemble (20 models, threshold-gated joining)

20 small attention LMs train independently. When val-accuracy crosses a band threshold
`[37, 47, 57, 67, 71, 73, 77, 79, 81, 85, 90]%`, the model **joins that band's pool**.
Inside a band there is a **grace period** (`GRACE_EPOCHS`) of isolated training before
cross-distillation / logit-averaging (collective bagging) is allowed. Rationale: early
cross-talk collapses diversity and bakes in shared bias; delayed joining preserves
independent errors so the bagged vote actually reduces variance/bias.

**BuddyUp mapping:** swap the synthetic corpus for `moderation_text` data (Reddit IRL +
profanity/Gen-Z slang) or `matching_embeddings` text towers. Export head -> ONNX like other notebooks.

In [1]:
import importlib.util, os, pathlib, sys
p = pathlib.Path(os.getcwd()).resolve()
ai = None
while p != p.parent:
    for cand in (p / 'backend' / 'ai_service', p / 'ai_service', p):
        if (cand / 'training').is_dir() and (cand / 'notebooks').is_dir():
            ai = cand; break
    if ai is not None: break
    p = p.parent
if ai is None: raise RuntimeError('ai_service not found')
sys.path.insert(0, str(ai / 'training')); sys.path.insert(0, str(ai))
os.chdir(ai / 'notebooks')
_missing = [m for m in ['torch'] if importlib.util.find_spec(m) is None]
if _missing:
    get_ipython().run_line_magic('pip', 'install -q ' + ' '.join(_missing))
try:
    from training.bootstrap import *; CFG = init()
except Exception as e:
    print('[bootstrap] unavailable:', e); CFG = {}
SCALE = CFG.get('scale', os.environ.get('BUDDY_SCALE', 'demo'))
print('scale:', SCALE)

2026-09-16 15:18:14.600226: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-16 15:18:14.742381: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


2026-09-16 15:18:18.716225: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


scale: smoke


In [2]:
import random, numpy as np, torch, torch.nn as nn
torch.manual_seed(42); np.random.seed(42); random.seed(42)
BANDS = [0.37, 0.47, 0.57, 0.67, 0.71, 0.73, 0.77, 0.79, 0.81, 0.85, 0.90]
GRACE_EPOCHS = {'smoke': 1, 'demo': 2, 'full': 5}[SCALE]
N_MODELS = {'smoke': 4, 'demo': 8, 'full': 20}[SCALE]  # 20 in full/GPU run
print(f'models={N_MODELS} bands={BANDS} grace={GRACE_EPOCHS}')

class TinyAttentionLM(nn.Module):
    """Embedding + 1 TransformerEncoderLayer + mean-pool + head. Stand-in for any attention NLP."""
    def __init__(self, vocab=2000, d=64, heads=4, nclass=2, seed=0):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        self.emb = nn.Embedding(vocab, d)
        nn.init.normal_(self.emb.weight, generator=g)
        enc = nn.TransformerEncoderLayer(d, heads, dim_feedforward=128, batch_first=True)
        # diversify init per member
        for p in enc.parameters(): nn.init.normal_(p, generator=g)
        self.enc = enc; self.head = nn.Linear(d, nclass)
        nn.init.normal_(self.head.weight, generator=g)
    def forward(self, x): return self.head(self.enc(self.emb(x)).mean(1))

class BandManager:
    def __init__(self, bands, grace):
        self.bands, self.grace = sorted(bands), grace
        self.acc = {}   # model_id -> best val acc
        self.joined = {}  # model_id -> (band, epoch_joined)
    def band_for(self, acc):
        b = None
        for t in self.bands:
            if acc >= t: b = t
        return b
    def update(self, mid, acc, epoch):
        self.acc[mid] = max(acc, self.acc.get(mid, 0.0))
        b = self.band_for(self.acc[mid])
        if b is not None and mid not in self.joined:
            self.joined[mid] = (b, epoch)
            print(f'  [band] model {mid} acc={self.acc[mid]:.3f} -> joins band {int(b*100)}% @epoch {epoch}')
        elif b is not None and b > self.joined[mid][0]:
            self.joined[mid] = (b, epoch)  # promote upward, grace restarts
            print(f'  [band] model {mid} acc={self.acc[mid]:.3f} -> PROMOTED to {int(b*100)}% @epoch {epoch}')
    def eligible(self, mid, epoch):
        """Allowed to distill only after grace epochs inside current band."""
        if mid not in self.joined: return False
        return (epoch - self.joined[mid][1]) >= self.grace
    def pools(self):
        out = {}
        for m, (b, _) in self.joined.items(): out.setdefault(b, []).append(m)
        return out

models=4 bands=[0.37, 0.47, 0.57, 0.67, 0.71, 0.73, 0.77, 0.79, 0.81, 0.85, 0.9] grace=1


In [3]:
# Synthetic binary sentiment corpus (stand-in for IMDB/AG-News/Reddit-IRL). Real path below.
VOCAB, SEQLEN = 2000, 32
def make_data(n, seed=0):
    rng = np.random.default_rng(seed)
    X = rng.integers(0, VOCAB, size=(n, SEQLEN))
    # planted signal: class 1 iff mean token id in top half + noise
    y = ((X.mean(1) > VOCAB/2) ^ (rng.random(n) < 0.15)).astype(np.int64)
    return torch.tensor(X), torch.tensor(y)
N = {'smoke': 2000, 'demo': 8000, 'full': 50000}[SCALE]
Xtr, ytr = make_data(N, 0); Xva, yva = make_data(2000, 1)
print('train', tuple(Xtr.shape), 'val', tuple(Xva.shape))
# REAL DATA (replace): datasets.load_dataset('imdb'|'ag_news'|'SetFit/SST2'), or
# BuddyUp Reddit-IRL via training/buddy_data.py; tokenize with a HF tokenizer, same loop.
# --- real batch (BUDDY_BATCH) overrides synthetic when present ---
from batch_data import has_batch, batch_meta, load_tensors
if has_batch():
    _m = batch_meta(); VOCAB, SEQLEN = _m['vocab'], _m['seqlen']
    _b = load_tensors('X', 'y', 'val_X', 'val_y')
    Xtr, ytr, Xva, yva = _b['X'], _b['y'], _b['val_X'], _b['val_y']
    print('REAL batch train', tuple(Xtr.shape), 'val', tuple(Xva.shape), '|', _m['source'])


train (2000, 32) val (2000, 32)
REAL batch train (28974, 32) val (5114, 32) | jokes(title)->0 + meirl(title)->1


In [4]:
EPOCHS = {'smoke': 2, 'demo': 5, 'full': 15}[SCALE]
models = [TinyAttentionLM(seed=i) for i in range(N_MODELS)]
opts = [torch.optim.Adam(m.parameters(), lr=3e-3) for m in models]
mgr = BandManager(BANDS, GRACE_EPOCHS)
ce = nn.CrossEntropyLoss()
BS = 256
for ep in range(EPOCHS):
    # 1) independent training step for every model
    for m, o in zip(models, opts):
        m.train(); perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), BS):
            idx = perm[i:i+BS]; o.zero_grad()
            loss = ce(m(Xtr[idx]), ytr[idx]); loss.backward(); o.step()
    # 2) evaluate + band assignment
    print(f'--- epoch {ep} ---')
    with torch.no_grad():
        for j, m in enumerate(models):
            m.eval()
            acc = (m(Xva).argmax(1) == yva).float().mean().item()
            mgr.update(j, acc, ep)
    # 3) collective bagging INSIDE each band, only for grace-eligible members:
    #    distill band-mean soft targets (T=2) with small weight; keeps diversity, cuts bias.
    with torch.no_grad():
        pools = mgr.pools()
    for b, members in pools.items():
        elig = [j for j in members if mgr.eligible(j, ep)]
        if len(elig) < 2: continue
        with torch.no_grad():
            logits = torch.stack([models[j](Xtr[:512]) for j in elig]).mean(0) / 2.0
            soft = torch.softmax(logits, -1)
        for j in elig:
            models[j].train(); opts[j].zero_grad()
            kd = -(soft * torch.log_softmax(models[j](Xtr[:512]) / 2.0, -1)).sum(-1).mean()
            (0.2 * kd).backward(); opts[j].step()
        print(f'  [distill] band {int(b*100)}%: {len(elig)}/{len(members)} eligible distilled')
print('final pools:', {int(k*100): v for k, v in mgr.pools().items()})

--- epoch 0 ---


  [band] model 0 acc=0.929 -> joins band 90% @epoch 0


  [band] model 1 acc=0.954 -> joins band 90% @epoch 0


  [band] model 2 acc=0.941 -> joins band 90% @epoch 0


  [band] model 3 acc=0.952 -> joins band 90% @epoch 0


--- epoch 1 ---


  [distill] band 90%: 4/4 eligible distilled
final pools: {90: [0, 1, 2, 3]}


In [5]:
# Evaluate: single-best vs band-bagged vote (bias/variance check)
import torch
with torch.no_grad():
    accs = [(m(Xva).argmax(1) == yva).float().mean().item() for m in models]
    for m in models: m.eval()
    vote = torch.stack([m(Xva) for m in models]).mean(0).argmax(1)
    bag = (vote == yva).float().mean().item()
print('member acc:', [round(a,3) for a in accs])
print(f'best single={max(accs):.3f}  bagged-ensemble={bag:.3f}')
# Disagreement = diversity proxy (want >0 before distill, shrinking after)
preds = torch.stack([(m(Xva).argmax(1)) for m in models]).numpy()
print('mean pairwise disagreement:', round((preds[None,:,:] != preds[:,None,:]).mean(), 3))
# Export best member to ONNX (matches ai_service serving contract)
best = models[int(np.argmax(accs))]; best.eval()
torch.onnx.export(best, torch.randint(0, VOCAB, (1, SEQLEN)), '../models/banded_nlp_best.onnx',
    input_names=['input_ids'], output_names=['logits'], dynamic_axes={'input_ids': {0: 'batch'}})
print('exported ../models/banded_nlp_best.onnx')

member acc: [0.956, 0.944, 0.955, 0.952]
best single=0.956  bagged-ensemble=0.963


mean pairwise disagreement: 0.033


/tmp/ipykernel_2834470/2882696240.py:15: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(best, torch.randint(0, VOCAB, (1, SEQLEN)), '../models/banded_nlp_best.onnx',


[torch.onnx] Obtain model graph for `TinyAttentionLM([...]` with `torch.export.export(..., strict=False)`...


[torch.onnx] Obtain model graph for `TinyAttentionLM([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...


[torch.onnx] Optimize the ONNX graph... ✅
exported ../models/banded_nlp_best.onnx


## Data, scrapers vs platforms (notebook 1)

| Option | What to use | Verdict |
|---|---|---|
| Public datasets (preferred start) | IMDB, AG News, SST-2, GLUE, Jigsaw toxicity, HuggingFace `SetFit/*` | Start here. Labelled, legal, reproducible. |
| BuddyUp first-party | `python manage.py export_ai_training_data` + `training/buddy_data.py` (Reddit-IRL + slang lexicons) | Best for prod moderation; needs volume before it beats public data. |
| Scrapers / internet bots | Common Crawl, custom Scrapy/Apify crawls | Only for *pretraining* corpora; labelling cost + copyright/PII risk. Prefer HF Datasets over raw scraping. |
| Managed platforms | HuggingFace AutoTrain / Together / Fireworks (fine-tune), W&B (tracking), LangSmith (eval) | Use for scale-up: train 20 members in parallel with one sweep config. No platform sells 'banded joining' off-the-shelf — this notebook IS the algorithm; platforms just run it. |

**Scale note:** 20 full LLMs is a GPU-cluster workload. Practice: 20 × LoRA adapters on one frozen base (PEFT) — same band logic, ~1% of the cost.